In [1]:
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np
import ast

In [3]:
def compute_common_authors(features_df, doi1, doi2):
    return len(set(features_df.loc[doi1, 'authors']).intersection(features_df.loc[doi2, 'authors']))

def compute_date_difference(features_df, doi1, doi2):
    date_format = "%Y-%m-%d"
    d1 = datetime.strptime(features_df.loc[doi1, 'publication_date'], date_format)
    d2 = datetime.strptime(features_df.loc[doi2, 'publication_date'], date_format)
    return abs((d1-d2).days)

def create_input_vector(features_df, doi1, doi2):
    return (ast.literal_eval(features_df.loc[doi1, 'title_sbert_embeddings']) + ast.literal_eval(features_df.loc[doi2, 'title_sbert_embeddings']) +
            [compute_common_authors(features_df, doi1, doi2), compute_date_difference(features_df, doi1, doi2)])

def create_labels(features_df, links_df, doi1, doi2):
    type1 = features_df.loc[doi1, 'paper_type']
    type2 = features_df.loc[doi2, 'paper_type']
    return int(len(links_df[(links_df[f'{type1}_doi'] == doi1) &
                            (links_df[f'{type2}_doi'] == doi2)]) == 1)

# test create_labels
# print(create_labels(features_df, links_df, '10.5334/johd.1', '10.1080/03071022.2014.975943'))
# print(create_labels(features_df, links_df, '10.5334/johd.4', '10.5334/johd.1'))

In [5]:
# ,doi,title,paper_type,concepts,authors,publication_date,title_sbert_embeddings
features_df = pd.read_csv('../data/paper-features.csv', index_col='doi')
links_df = pd.read_csv('../data/links.csv')

In [10]:
print(links_df.shape)
print(len(set(links_df['data_paper_doi']).union(set(links_df['research_paper_doi']))))
links_df.head(2)

(80, 9)
152


,Unnamed: 0,data_paper_doi,research_paper_doi,data_paper_url,research_paper_url,research_paper_file_url,data_paper_has_file,research_paper_has_file,source
0,0,10.5334/johd.1,10.1080/03071022.2014.975943,https://openhumanitiesdata.metajnl.com/article...,https://www.tandfonline.com/doi/full/10.1080/0...,http://sro.sussex.ac.uk/id/eprint/51704/1/RSHI...,True,True,original-johd
1,1,10.5334/johd.1,10.1080/01615440.2015.1007194,https://openhumanitiesdata.metajnl.com/article...,https://www.tandfonline.com/doi/full/10.1080/0...,https://discovery.ucl.ac.uk/10105667/1/Irish%2...,True,True,original-johd


In [7]:
print(features_df.shape)
features_df.head(2)

(148, 7)


,Unnamed: 0,title,paper_type,concepts,authors,publication_date,title_sbert_embeddings
doi,,,,,,,
10.1002/asi.24499,0,Harmonizing and publishing heterogeneous premo...,research_paper,"['https://openalex.org/A5010723046', 'https://...","['https://openalex.org/A5010723046', 'https://...",2021-05-26,"[-0.3494153618812561, 0.05191454291343689, -0...."
10.1002/hipo.23539,1,Single neurons in the human medial temporal lo...,research_paper,"['https://openalex.org/A5034612103', 'https://...","['https://openalex.org/A5034612103', 'https://...",2023-04-15,"[0.35188332200050354, -0.49159741401672363, -0..."


In [11]:
# Splitting data into training and testing
# X_train, X_test, y_train, y_test = train_test_split(df, df['paper_type'], test_size=0.2, stratify=df['paper_type'])
all_data_orig = []
for doi1 in features_df.index:
    for doi2 in features_df.index:
        if doi1 == doi2:
            continue
        all_data_orig.append((create_input_vector(features_df, doi1, doi2), 
                            create_labels(features_df, links_df, doi1, doi2)))

In [12]:
all_data_xs = torch.tensor([a[0] for a in all_data_orig])
all_data_ys = torch.tensor([a[1] for a in all_data_orig], dtype=torch.float)
mean = all_data_xs.mean(dim=0)
std = all_data_xs.std(dim=0)
all_data_xs = (all_data_xs - mean) / std # normalize (convert to z-scores)

In [17]:
all_data_xs.shape # ~= 22k ~= 150*150

torch.Size([21756, 770])

In [13]:
all_data = pd.DataFrame({'x': [x for x in all_data_xs], 'y': all_data_ys})

In [14]:
print(all_data.shape)
print(all_data.iloc[0,:])
print(all_data.iloc[0,0])

(21756, 2)
x    [tensor(-0.8719), tensor(0.1038), tensor(-0.36...
y                                                  0.0
Name: 0, dtype: object
tensor([-8.7190e-01,  1.0382e-01, -3.6415e-01, -4.6531e-01,  2.1592e-01,
         3.5139e-01, -2.0454e+00,  8.8322e-01,  2.2723e-01, -7.3340e-01,
         2.7582e-01,  8.7743e-01, -3.2823e-01, -3.8814e-01,  2.1121e-02,
        -1.3024e-01,  1.6216e-01,  5.1751e-01, -1.2578e-01,  3.0445e-01,
        -1.0728e+00,  9.8676e-01,  2.3396e-02, -1.1142e+00, -1.2762e-01,
        -5.1336e-01, -9.8808e-01, -3.5440e-01,  1.2028e+00, -1.1432e+00,
         5.7619e-01,  2.2760e-01,  6.5965e-01,  3.2077e-01,  1.0211e-01,
        -7.8394e-01, -3.0697e-01,  2.2297e-01, -9.3478e-01, -3.3195e-01,
         1.0131e-01,  5.4721e-01, -1.5130e+00,  1.5290e+00, -2.8460e-01,
        -5.9774e-01, -7.9322e-01,  1.2300e-01, -2.0930e-01,  1.1203e+00,
        -5.3162e-01,  1.2684e-01, -1.6843e+00,  5.3717e-01,  8.6158e-01,
         4.6147e-01, -2.7514e-01, -4.2944e-01, -1.544

In [15]:
all_data_pos = all_data[all_data['y'] == 1]

# as negative samples are way more than positive samples, we sample 2x positive samples 
all_data_neg = all_data[all_data['y'] == 0].sample(2*len(all_data_pos))
all_data = pd.concat([all_data_pos, all_data_neg]).sample(frac=1.0)

In [18]:
train_data = all_data[0:2*len(all_data)//3]
test_data = all_data[2*len(all_data)//3:]
print(train_data.shape, test_data.shape)

(304, 2) (152, 2)


In [19]:
# Model Definition
class PaperRelationClassifier(nn.Module):
    def __init__(self):
        super(PaperRelationClassifier, self).__init__()
        self.fc1 = nn.Linear(770, 128)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 64)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(64, 1)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout1(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        x = torch.sigmoid(x)
        return x

In [20]:
# Model Training
from tqdm import tqdm

model = PaperRelationClassifier()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 5

for epoch in tqdm(range(num_epochs)):
    for inputs, label in train_data.itertuples(index=False):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output.squeeze(), torch.tensor(label))
        loss.backward()
        optimizer.step()


c:\Users\k21191796\Downloads\wr\nlp-automatic-linking\nlp-automatic-linking-proj\.conda\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 5/5 [00:03<00:00,  1.52it/s]


In [21]:
# Model Evaluation
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for inputs, label in test_data.itertuples(index=False):
        output = model(inputs)
        pred = (output.squeeze() > 0.5).float()
        all_preds.append(pred.numpy())
        all_labels.append(label)

accuracy = accuracy_score(all_labels, all_preds)
print(f'Accuracy: {accuracy * 100:.2f}%')
print('Classification Report:')
print(classification_report(all_labels, all_preds))

Accuracy: 62.50%
Classification Report:
              precision    recall  f1-score   support

         0.0       0.70      0.81      0.75       105
         1.0       0.33      0.21      0.26        47

    accuracy                           0.62       152
   macro avg       0.52      0.51      0.50       152
weighted avg       0.58      0.62      0.60       152

